# NumPy practice: shape, broadcasting, vectorized math

Use the **CS224N (.venv)** kernel. Run the cells from top to bottom, and stop when a cell asks you to predict.

A shape is the size of each axis. `@` does the loop inside NumPy. Broadcasting lines shapes up from the right and stretches an axis of length 1.

In [1]:
import numpy as np

words = np.array(
    [
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.2, 0.3, 0.9],
    ]
)
query = np.array([0.2, -0.1, 0.5])
bias = np.array([10.0, 20.0, 30.0])
column = np.array([[1.0], [2.0], [3.0]])
row = np.array([10.0, 20.0])

words.shape, query.shape, bias.shape, column.shape, row.shape

((3, 3), (3,), (3,), (3, 1), (2,))

## Predict before you compute

Replace each `None` with a shape tuple, such as `(3,)` or `(3, 2)`. Do this before you run the next code cell.

In [2]:
predicted_shapes = {
    "words": (3,3),
    "query": (3,),
    "words @ query": (3,1),
    "words + bias": (3,3),
    "column + row": (3,2),
}

## Look at the results

Run this only after the predictions are filled in. Each expression prints the array and then its shape.

In [3]:
print("words")
print(words)
print("query", query)
print("words @ query", words @ query)
print("words + bias")
print(words + bias)
print("column + row")
print(column + row)

words
[[1.  0.  0. ]
 [0.  1.  0. ]
 [0.2 0.3 0.9]]
query [ 0.2 -0.1  0.5]
words @ query [ 0.2  -0.1   0.46]
words + bias
[[11.  20.  30. ]
 [10.  21.  30. ]
 [10.2 20.3 30.9]]
column + row
[[11. 21.]
 [12. 22.]
 [13. 23.]]


In [4]:
actual_shapes = {
    "words": words.shape,
    "query": query.shape,
    "words @ query": (words @ query).shape,
    "words + bias": (words + bias).shape,
    "column + row": (column + row).shape,
}

for name, actual in actual_shapes.items():
    guessed = predicted_shapes[name]
    if guessed is None:
        print(f"{name}: actual {actual}  (fill in your prediction)")
    elif tuple(guessed) == tuple(actual):
        print(f"{name}: {actual}  correct")
    else:
        print(f"{name}: you said {guessed}, actual {actual}")

words: (3, 3)  correct
query: (3,)  correct
words @ query: you said (3, 1), actual (3,)
words + bias: (3, 3)  correct
column + row: (3, 2)  correct


## Two different rules

Your four broadcasting predictions followed the elementwise rule. `words @ query` did not, because `@` is a different operation.

`(3,)` and `(3, 1)` are different objects.

- `(3,)` is a 1-D array. It is neither a row nor a column.
- `(1, 3)` is a row, a 2-D array with one row.
- `(3, 1)` is a column, a 2-D array with one column.

`@` does not add an axis. A 1-D vector in, a 1-D vector out. That is why `(3, 3) @ (3,)` is `(3,)`, three scores, and not a column of shape `(3, 1)`.


In [ ]:
print("query", query.shape)
print("as a column", query.reshape(3, 1).shape)
print("words @ query", (words @ query).shape)
print("words @ column", (words @ query.reshape(3, 1)).shape)


## Rule 1. `@` multiplies and removes the shared axis

`@` does not stretch a mismatched axis. The inner sizes must be equal. NumPy multiplies along that shared axis and drops it from the result. If those inner sizes differ, `@` raises an error.

Write the two shapes. The shared axis is the **last** axis of the left operand and the **first** axis of the right operand, except that a 1-D array is entirely that shared axis.

| Left | Right | Result |
| --- | --- | --- |
| `(k,)` | `(k,)` | a scalar, shape `()` |
| `(n, k)` | `(k,)` | `(n,)` |
| `(k,)` | `(k, m)` | `(m,)` |
| `(n, k)` | `(k, m)` | `(n, m)` |

`words @ query` is the second row: `n = 3`, `k = 3`, and the query has no axis left over, so the result is `(3,)`.

`words @ query.reshape(3, 1)` is the fourth row: `(3, 3) @ (3, 1)` gives `(3, 1)`. The extra axis was already on the column. `@` kept it.

## Rule 2. `+`, `-`, `*`, and `/` broadcast

These operations line the shapes up from the **right** and stretch axes of length 1. They do not require the shapes to be equal.

1. Write both shapes.
2. If one shape has fewer axes, add axes of length 1 on its **left** until both have the same number of axes.
3. Compare one axis at a time. The two sizes must be equal, or one of them must be 1.
4. A size of 1 stretches to the other size. The result takes that larger size.
5. If both sizes are greater than 1 and they differ, the operation raises an error. Nothing is stretched.

`words + bias`, shapes `(3, 3)` and `(3,)`:

1. Pad `(3,)` on the left to `(1, 3)`.
2. Axis 0: `3` and `1` → `3`. Axis 1: `3` and `3` → `3`.
3. Result `(3, 3)`. The bias row is copied onto every word.

`column + row`, shapes `(3, 1)` and `(2,)`:

1. Pad `(2,)` on the left to `(1, 2)`.
2. Axis 0: `3` and `1` → `3`. Axis 1: `1` and `2` → `2`.
3. Result `(3, 2)`.

Same idea for `*` and `/`. The operation is still elementwise after the stretch. `@` never uses this procedure.


## Apply the rule

Fill these in before you run the check. Use Rule 1 for `@` and Rule 2 for `+`. One of them is an error: write `"error"` for that one.


In [ ]:
rule_predictions = {
    "(4, 5) @ (5, 2)": None,
    "(4, 5) @ (5,)": None,
    "(5,) @ (5, 2)": None,
    "(4, 3) + (3,)": None,
    "(4, 1) + (3,)": None,
    "(4, 3) + (4,)": None,
}


In [ ]:
import numpy as np

examples = {
    "(4, 5) @ (5, 2)": lambda: np.zeros((4, 5)) @ np.zeros((5, 2)),
    "(4, 5) @ (5,)": lambda: np.zeros((4, 5)) @ np.zeros((5,)),
    "(5,) @ (5, 2)": lambda: np.zeros((5,)) @ np.zeros((5, 2)),
    "(4, 3) + (3,)": lambda: np.zeros((4, 3)) + np.zeros((3,)),
    "(4, 1) + (3,)": lambda: np.zeros((4, 1)) + np.zeros((3,)),
    "(4, 3) + (4,)": lambda: np.zeros((4, 3)) + np.zeros((4,)),
}

for name, compute in examples.items():
    guessed = rule_predictions[name]
    try:
        actual = compute().shape
    except ValueError:
        actual = "error"
    if guessed is None:
        print(f"{name}: actual {actual}  (fill in your prediction)")
    elif guessed == actual or (guessed == "error" and actual == "error"):
        print(f"{name}: {actual}  correct")
    else:
        print(f"{name}: you said {guessed}, actual {actual}")


## 3-D, 4-D, and higher

The 2-D tables are the special case of one rule. Extra axes sit on the **left**. They are a stack, usually a batch of sentences or a batch of examples. NumPy does not invent a new rule for each rank.

### `@` on stacks

Split every operand into a batch and a matrix.

- The **matrix** is the last two axes. A 1-D operand is still the exception from Rule 1: on the right it behaves like a column that is removed afterward, and on the left it behaves like a row that is removed afterward.
- The **batch** is everything to the left of those matrix axes.
- Multiply the matrices with Rule 1. The inner sizes still must match. `@` does not stretch them.
- Combine the batches with Rule 2. Those leading axes **do** broadcast: pad on the left, then stretch a 1.

`(2, 3, 4) @ (4, 5)`:

- Left matrix `(3, 4)`, right matrix `(4, 5)`, shared `4`. Matrix result `(3, 5)`.
- Left batch `(2,)`. The right array is only a matrix, so its batch is empty and pads to `(1,)`.
- Batch `2` and `1` → `2`.
- Result `(2, 3, 5)`. Two separate `(3, 4) @ (4, 5)` products.

`(2, 3, 4) @ (4,)`:

- The right operand is 1-D, so Rule 1 gives `(3, 4) @ (4,) → (3,)`.
- The batch `(2,)` stays.
- Result `(2, 3)`.

`(8, 2, 3, 4) @ (1, 4, 5)`:

- Matrices `(3, 4) @ (4, 5) → (3, 5)`.
- Batches `(8, 2)` and `(1,)`. Pad the right batch to `(1, 1)`.
- `8` and `1` → `8`. `2` and `1` → `2`.
- Result `(8, 2, 3, 5)`.

`(2, 3, 4) @ (3, 4, 5)` is an error. The matrices match, `(3, 4) @ (4, 5)`. The batches are `(2,)` and `(3,)`. Both are greater than 1 and they differ, so nothing stretches.

### `+` and `*` on stacks

Rule 2 does not change. Align from the right, pad 1s on the left, stretch a 1, and error when two sizes greater than 1 disagree.

`(2, 3, 4) + (4,)` pads the right shape to `(1, 1, 4)`, then stretches to `(2, 3, 4)`. The length-4 vector is added to every row of every matrix in the stack.

`(2, 1, 4) + (3, 1)` pads `(3, 1)` to `(1, 3, 1)`. Then `(2, 1, 4)` against `(1, 3, 1)` becomes `(2, 3, 4)`.


## Apply the rule to stacks

Fill these in before you run the check. One of them is an error: write `"error"` for that one.


In [ ]:
stack_predictions = {
    "(2, 3, 4) @ (4, 5)": None,
    "(2, 3, 4) @ (4,)": None,
    "(8, 2, 3, 4) @ (1, 4, 5)": None,
    "(2, 3, 4) @ (3, 4, 5)": None,
    "(2, 3, 4) + (4,)": None,
    "(2, 1, 4) + (3, 1)": None,
}


In [ ]:
stack_examples = {
    "(2, 3, 4) @ (4, 5)": lambda: np.zeros((2, 3, 4)) @ np.zeros((4, 5)),
    "(2, 3, 4) @ (4,)": lambda: np.zeros((2, 3, 4)) @ np.zeros((4,)),
    "(8, 2, 3, 4) @ (1, 4, 5)": lambda: np.zeros((8, 2, 3, 4)) @ np.zeros((1, 4, 5)),
    "(2, 3, 4) @ (3, 4, 5)": lambda: np.zeros((2, 3, 4)) @ np.zeros((3, 4, 5)),
    "(2, 3, 4) + (4,)": lambda: np.zeros((2, 3, 4)) + np.zeros((4,)),
    "(2, 1, 4) + (3, 1)": lambda: np.zeros((2, 1, 4)) + np.zeros((3, 1)),
}

for name, compute in stack_examples.items():
    guessed = stack_predictions[name]
    try:
        actual = compute().shape
    except ValueError:
        actual = "error"
    if guessed is None:
        print(f"{name}: actual {actual}  (fill in your prediction)")
    elif guessed == actual or (guessed == "error" and actual == "error"):
        print(f"{name}: {actual}  correct")
    else:
        print(f"{name}: you said {guessed}, actual {actual}")
